In [1]:
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile

import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import itertools
import random
import numpy as np
import json
import shutil

In [4]:

pretrain_modality = 'standard'  # 'standard', 'contrastive
train_modality = 'fine_tune' # 'fine_tune','progressive','from_scratch'
selected_model = 'resnet18' #'DeiT-Tiny'  # Example model, can be changed
type_of_model = 'CNN'  # 'CNN', 'Transformer'
huggingface = False  # Set to True if using Hugging Face models

selected_classifier = 'MLPClassifier1' #'logreg'
pretrained = False if train_modality == 'from_scratch' else True
use_external_validation = True  # Set to True if using an external validation set
loss_criterion = 'NTXentLoss' if pretrain_modality=='contrastive' else 'CrossEntropyLoss'
val_percentage = 0.2 if pretrain_modality == 'contrastive' else 1.0
use_amp = False

use_external_validation = True if pretrain_modality == 'contrastive' else use_external_validation

output_dir = source_path+f'\\outputs\\test\\{selected_model}\\{train_modality}'
file_IO.access_or_create_dir(output_dir)
script_name = source_path+"/scripts/fine-tuning.py"
kind = 'patches_224'  # Example kind, can be changed body, contrastive, patches_224
input_preprocessed=file_IO.load_preprocessed_files(kind, mode='train')
val_filename = file_IO.load_preprocessed_files(kind, 'val')

model_mode = 'truncated'  # 'truncation', 'full', 'truncated'
truncation='remove head'
custom_transform = False  # Set to True for custom transforms
transform_mode = 'resize'  # 'train', 'val', 'test', 'resize'
use_augmentation = False  # Set to True for data augmentation
n_sub_patches=-1

saved='old-laptop'  # 'new', 'old-laptop', 'new-laptop'
train_df = pd.read_csv(source_path+f'\\outputs\\preprocessed_data\\{input_preprocessed}')
if train_df['file_name'][0].startswith('C'):
    saved = 'new-laptop'

In [ ]:
clear_directory = True  # Set to True to clear the directory before saving checkpoints
total_epochs = 1  # Total number of epochs for fine-tuning
patience=total_epochs

save_path = output_dir
file_IO.access_or_create_dir(save_path)
checkpoint_path=save_path+'\\checkpoints'
file_IO.access_or_create_dir(checkpoint_path)
if clear_directory==True:
    print(f'Clearing directory: {checkpoint_path}')
    file_IO.clear_folder(checkpoint_path)

log_grad_norm = False
optim_config = {
            'optimizer_phases':[total_epochs],  # Example: [10, 10, 80] for 100 epochs
            'phase_layers_to_freeze':[[]],
            'phase_scheduling': ['no_scheduling'],
            'phase_optimizer':['Adam'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
            'phase_lr': [1e-4],
            'phase_optimizer_hyperparams': [{}],
            'phase_scheduler_hyperparams': [{}],
        }
nn_parameters={
    'dropout': 0.2,
    'n_neurons': 10,
    'activation': 'relu',  # Default to 'relu' if not specified
    'with_input_norm': None,  # Default to True if not specified
}

N_max = 60000//(40*4)

args = script_launching.DotDict(
    N_max=-1,
    patches=True,
    input_filename=input_preprocessed,
    val_filename=val_filename if use_external_validation else None,
    huggingface=huggingface,
    pooling=False,  # if true in transformer models use pooling, if false only the cls token
    custom_transform=custom_transform,  # custom transform for the dataset
    transform_mode=transform_mode,  # 'train', 'val', 'test'
    save_h5=False,
    selected_model=selected_model,  # googlenet, alexnet
    selected_classifier=selected_classifier,  # 'logreg', 'svm', 'rf', 'gbc', 'mlp', 'dt'
    truncation=truncation,
    running='new-laptop',
    saved=saved,
    model_mode=model_mode,  # 'truncation
    batch_size=64,
    select_cls=False,
    num_workers=4,
    pin_memory=True,
    show_image=False,
    checkpoint_path = checkpoint_path+"\\checkpoint.pt",
    save_path = save_path,
    total_epochs = total_epochs,
    log_grad_norm = log_grad_norm,
    use_profiler = False,
    run_epochs = total_epochs,
    plot_every = 1,
    patience = patience,
    use_amp = use_amp ,#mixed precision training,
    val_percentage= val_percentage ,#percentage of validation data used for linear evaluation,
    n_splits = 4,
    loss_criterion = loss_criterion,
    optim_config = optim_config,
    use_augmentation=use_augmentation,  # Set to True for data augmentation
    n_patches=n_sub_patches,
    contrastive_mode=False,  # Set to True for contrastive learning
    load_contrastive=False,  # Set to True if loading a trained contrastive model for fine tuning
    nn_parameters=nn_parameters,  # Dictionary with neural network parameters
    type_of_search='single_search',  # 'grid_search', 'single_search'
    train_modality=train_modality,  # 'fine_tune', 'progressive', 'from_scratch'
    pretrain_modality=pretrain_modality,  # 'standard', 'contrastive'
    load_data_from='zarr',  # 'zarr', 'folder'
)
script_launching.run_experiment_threaded(args,script_name)  # Test a single run first

Clearing directory: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\test\resnet18\fine_tune\checkpoints
Starting experiment:
[STDOUT] Output shape:  torch.Size([1, 512])
[STDOUT] tensor([[-0.3723,  0.4593]])
[STDOUT] Device is:  cuda
[STDOUT] Train dataset size: 56400
[STDOUT] Validation dataset size: 14200
[STDOUT] [GPU Memory] Allocated: 0.00 MB | Reserved: 0.00 MB
[STDOUT] Model size: 44.81 MB
[STDOUT] 📂 No checkpoint found at c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\test\resnet18\fine_tune\checkpoints\checkpoint.pt. Starting fresh training.
[STDOUT] Setting optimizer for phase 0: Adam with learning rate 0.0001
[STDOUT] Freezing [] parameters
[STDOUT] Trainable parameters after freezing: 11,181,664
[STDOUT] 
[STDOUT] Epoch 0 - Optimization Phase: 0
[STDERR] 
[STDERR] Epoch 1/1 [Train]:   2%|▏         | 19/882 [01:23<26:27,  1.84s/it]


result: 64 in 2s -> 32 al s -> un fattore 10 piu' lento nonostante il parallelismo etc ..
pre-processed: 64*2 o 64*3 al secondo

# reload

In [3]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import  utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()